In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms
import pathlib
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==========================================
# 1. REPRODUCIBILITY
# ==========================================
def set_seed(seed=42):
    """Fix all random seeds to ensure fully reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==========================================
# 2. CONFIGURATION
# ==========================================
IMAGE_SIZE   = 224       # DO NOT CHANGE – required by project specification
BATCH_SIZE   = 64
EPOCHS       = 150       # Tăng lên 150 epoch để mô hình có đủ thời gian hội tụ sâu hơn
MAX_LR       = 2e-3      # Peak learning rate
DEVICE       = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. DATA PREPARATION & AUGMENTATION
# ==========================================
data_dir = pathlib.Path(
    "/kaggle/input/datasets/huynhthethien/radarcommunsignaldata2026train"
)

# Load base dataset without transform to inspect metadata
base_dataset = datasets.ImageFolder(root=str(data_dir))
num_classes  = len(base_dataset.classes)
class_names  = base_dataset.classes
print(f"Total images  : {len(base_dataset)}")
print(f"Num classes   : {num_classes}")
print(f"Classes       : {class_names}")

# 80/20 train-validation split with fixed seed
train_size = int(0.8 * len(base_dataset))
val_size   = len(base_dataset) - train_size
generator  = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(
    base_dataset, [train_size, val_size], generator=generator
)
print(f"Train samples : {train_size}")
print(f"Val samples   : {val_size}")

class CustomSubset(Dataset):
    """Wrap a random_split Subset and apply a per-split transform."""
    def __init__(self, subset, transform=None):
        self.subset    = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# ── Training Transform ────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    # Đã gỡ bỏ RandomAffine để không làm sai lệch pha của các nhóm QAM/PSK
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), value=0), 
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# ── Validation Transform ──────────────────────────────────────────────────────
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = CustomSubset(train_subset, transform=train_transform)
val_dataset   = CustomSubset(val_subset,   transform=val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel-wise attention."""
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.fc1     = nn.Linear(channels, mid, bias=False)
        self.relu    = nn.ReLU(inplace=True)
        self.fc2     = nn.Linear(mid, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.sigmoid(self.fc2(self.relu(self.fc1(s)))).view(b, c, 1, 1)
        return x * s

class InvertedResidualSE(nn.Module):
    """MobileNetV2 Inverted Residual block with SE attention."""
    def __init__(self, inp: int, oup: int, stride: int, expand_ratio: int, use_se: bool = True, se_reduction: int = 4):
        super().__init__()
        hidden = round(inp * expand_ratio)
        self.use_res_connect = (stride == 1 and inp == oup)

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(inp, hidden, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden),
                nn.ReLU6(inplace=True),
            ]
        layers += [
            nn.Conv2d(hidden, hidden, kernel_size=3, stride=stride, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),
            nn.Conv2d(hidden, oup, kernel_size=1, bias=False),
            nn.BatchNorm2d(oup),
        ]
        self.conv = nn.Sequential(*layers)
        self.se   = SEBlock(oup, se_reduction) if use_se else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.se(self.conv(x))
        if self.use_res_connect:
            return x + out
        return out

class BasicCNN(nn.Module):
    """
    KIẾN TRÚC MÔ HÌNH: Đã đổi tên chuẩn thành BasicCNN theo yêu cầu.
    """
    def __init__(self, num_classes: int = 12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU6(inplace=True),
        )

        self.blocks = nn.Sequential(
            InvertedResidualSE(16, 16, stride=1, expand_ratio=1, use_se=False),
            InvertedResidualSE(16, 24, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(24, 24, stride=1, expand_ratio=6, use_se=True),
            InvertedResidualSE(24, 32, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(32, 32, stride=1, expand_ratio=6, use_se=True),
            InvertedResidualSE(32, 48, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(48, 64, stride=2, expand_ratio=4, use_se=True),
        )

        self.head = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.AdaptiveAvgPool2d(1),   
        )

        self.dropout    = nn.Dropout(0.35)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)            
        x = self.blocks(x)          
        x = self.head(x)            
        x = x.view(x.size(0), -1)  
        x = self.dropout(x)
        x = self.classifier(x)      
        return x

# ── Instantiate and verify parameter budget ───────────────────────────────────
model = BasicCNN(num_classes).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n[INFO] Total trainable parameters: {total_params:,}")
assert total_params < 100_000, f"Parameter budget exceeded: {total_params:,} >= 100,000"
print(f"[INFO] Parameter constraint satisfied: {total_params:,} < 100,000 ✓")

# ==========================================
# 5. TRAINING SETUP
# ==========================================

# Giảm label smoothing về 0.05 để mô hình phân loại tự tin hơn
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Giảm weight_decay về 0.02 để mô hình có không gian học tốt hơn
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.02)

steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    pct_start=0.15, 
    div_factor=10,
    final_div_factor=1000,
)

history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc":  [],
}

# ==========================================
# 6. TRAINING LOOP
# ==========================================

def train_model(model, train_loader, val_loader):
    best_val_acc     = 0.0
    final_val_preds  = []
    final_val_labels = []
    best_state_dict  = None

    print("\n[INFO] Starting training...")
    print("-" * 78)

    for epoch in range(EPOCHS):

        # ── TRAINING PHASE ────────────────────────────────────────────────
        model.train()
        running_loss      = 0.0
        all_train_preds   = []
        all_train_labels  = []

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        train_acc  = accuracy_score(all_train_labels, all_train_preds)

        # ── VALIDATION PHASE ──────────────────────────────────────────────
        model.eval()
        val_running_loss  = 0.0
        all_val_preds     = []
        all_val_labels    = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = model(images)
                loss    = criterion(outputs, labels)
                
                val_running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader)
        val_acc  = accuracy_score(all_val_labels, all_val_preds)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            final_val_preds  = all_val_preds
            final_val_labels = all_val_labels
            best_state_dict  = copy.deepcopy(model.state_dict())

        # In log chi tiết ở MỖI epoch
        print(
            f"Epoch [{epoch + 1:03d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
            f"Best Val: {best_val_acc:.4f}"
        )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print(f"\n[INFO] Restored best checkpoint -> Val Acc = {best_val_acc:.4f}")

    return final_val_labels, final_val_preds

final_labels, final_preds = train_model(model, train_loader, val_loader)

# ==========================================
# 7. EVALUATION METRICS
# ==========================================
print("\n" + "=" * 78)
print("[INFO] DETAILED CLASSIFICATION REPORT")
print("=" * 78)

overall_acc = accuracy_score(final_labels, final_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    final_labels, final_preds, zero_division=0
)

print(f"Overall Accuracy: {overall_acc:.4f}\n")
hdr = (f"{'Class':<12}  {'Precision':>10}  {'Recall':>10}  {'F1-score':>10}  {'Support':>8}")
print(hdr)
print("-" * len(hdr))
for i, cls in enumerate(class_names):
    print(
        f"{cls:<12}  {precision[i]:>10.4f}  {recall[i]:>10.4f}"
        f"  {f1[i]:>10.4f}  {int(support[i]):>8}"
    )
print("=" * 78)

# ==========================================
# 8. PLOTTING
# ==========================================
epochs_x = range(1, EPOCHS + 1)

# Training/Val Loss
plt.figure(figsize=(9, 5))
plt.plot(epochs_x, history["train_loss"], label="Training Loss",   linewidth=2)
plt.plot(epochs_x, history["val_loss"],   label="Validation Loss", linewidth=2)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Loss",  fontsize=13)
plt.title("Training and Validation Loss", fontsize=15)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("loss_plot.png", dpi=300)
plt.close()

# Training/Val Accuracy
plt.figure(figsize=(9, 5))
plt.plot(epochs_x, history["train_acc"], label="Training Accuracy",   linewidth=2)
plt.plot(epochs_x, history["val_acc"],   label="Validation Accuracy", linewidth=2)
plt.axhline(y=0.90, color="red", linestyle="--", alpha=0.7, label="90% target")
plt.xlabel("Epoch",    fontsize=13)
plt.ylabel("Accuracy", fontsize=13)
plt.title("Training and Validation Accuracy", fontsize=15)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("accuracy_plot.png", dpi=300)
plt.close()

# Confusion Matrix
cm = confusion_matrix(final_labels, final_preds)
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5,
)
plt.xlabel("Predicted Label", fontsize=13)
plt.ylabel("True Label",      fontsize=13)
plt.title("Confusion Matrix", fontsize=15)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.close()
print("[INFO] Saved all plots successfully.")

# ==========================================
# 9. MODEL EXPORT
# ==========================================
############################################
# DO NOT MODIFY THIS SECTION
############################################
model.eval()
example_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
traced_model  = torch.jit.trace(model, example_input)
GroupID       = "04"
model_name    = f"{GroupID}_DeepLearning Project_TrainedModel.pt"
traced_model.save(model_name)
print("Model saved:", model_name)

Device: cuda:0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total images  : 76800
Num classes   : 12
Classes       : ['16-QAM', 'B-FM', 'BPSK', 'Barker', 'CPFSK', 'DSB-AM', 'GFSK', 'LFM', 'PAM4', 'QPSK', 'Rect', 'StepFM']
Train samples : 61440
Val samples   : 15360

[INFO] Total trainable parameters: 96,812
[INFO] Parameter constraint satisfied: 96,812 < 100,000 ✓

[INFO] Starting training...
------------------------------------------------------------------------------
Epoch [001/150] | Train Loss: 1.5977  Acc: 0.4686 | Val Loss: 1.0757  Acc: 0.6632 | Best Val: 0.6632
Epoch [002/150] | Train Loss: 1.0225  Acc: 0.6883 | Val Loss: 0.8572  Acc: 0.7568 | Best Val: 0.7568
Epoch [003/150] | Train Loss: 0.8588  Acc: 0.7632 | Val Loss: 0.7317  Acc: 0.8109 | Best Val: 0.8109
Epoch [004/150] | Train Loss: 0.7819  Acc: 0.7968 | Val Loss: 0.6972  Acc: 0.8271 | Best Val: 0.8271
Epoch [005/150] | Train Loss: 0.7409  Acc: 0.8150 | Val Loss: 0.6853  Acc: 0.8258 | Best Val: 0.8271
Epoch [006/150]